# Abstention and Coverage (Inline)


In [ ]:
import os, sys
sys.path.insert(0, os.environ.get("GMS_RAG_TUTORIAL", "/home/asudjianto/jupyterlab/agent-tutorial-private/beyond-chunk-and-pray/code"))
from forgeloop.rag import load_store, CORPUS, STORE, EVAL_COHORT, DEVICE, store_config

# Chapter 11 — Honest abstention and coverage

A trustworthy RAG system has to know, and *say*, what it does not know.
Triple-mediated retrieval gives us a hard, measurable notion of *don't know*:
if a question does not bind to a known entity/relation, or no grounded fact
matches, the pipeline **abstains** instead of guessing. And because every
answerable fact came from a triple with provenance, the regions of the
document that carry **body text but no triples** are *measurable blind
spots* — not a silent failure mode.

This chapter shows three things over the Northwind FY2025 report:
1. a prose (MD&A / Risk / Outlook) question **abstains with a notice**;
2. `coverage_report(store)` flags exactly the prose sections as blind spots;
3. `RagAnswer.audit()` emits a per-query audit trail.

## The corpus and its store

We use the shipped Northwind FY2025 report and its trained store. The store
was built by `scripts/build_store.py` (see Foundation brief F2); we never
rebuild it here.

In [ ]:
REPORT_MD = CORPUS
STORE_PATH = STORE
with open(REPORT_MD) as f:
    markdown = f.read()
print(f"report: {REPORT_MD}")
print(f"store:  {STORE_PATH}")

## Listing 2 — the coverage monitor

`coverage_report` walks the document section by section, buckets each
content triple by the source line its provenance resolves to, and reports
which sections have body text but zero triples. Those are the blind spots a
triple-mediated pipeline cannot reach.

We can compute coverage from just the markdown plus the store's triple list
— no model forward pass — so this cell runs on CPU. The trained store ships
its triple list as `triples.json` on disk; we read it directly and wrap it
in a tiny stand-in object carrying `.markdown` and `.triples`, which is all
`coverage_report` reads. In production you pass the real `GMSExpertStore`.

In [ ]:
import json
from dataclasses import dataclass, field
from knowlytix.knowledge.rag import coverage_report

# The trained store's triple list, read from disk (CPU-only, no model load).
with open(os.path.join(STORE_PATH, "triples.json")) as f:
    TRIPLES = [tuple(t) for t in json.load(f)]
print(f"{len(TRIPLES)} triples loaded from the store (incl. structural in_section)")

@dataclass
class _StoreView:
    """Minimal view coverage_report needs: .markdown and .triples."""
    markdown: str
    triples: list = field(default_factory=list)
    store_path: str = ""

store_view = _StoreView(markdown=markdown, triples=TRIPLES, store_path=STORE_PATH)
report = coverage_report(store_view)

print(f"coverage_ratio = {report.coverage_ratio:.2f}")
for r in report.regions:
    mark = "x" if r.covered else " "
    flag = "  <-- BLIND SPOT" if r.blind_spot else ""
    print(f"[{mark}] {r.title} ({r.triple_count} triples, "
          f"{r.body_lines} body lines){flag}")
print()
print("blind spots:", [r.title for r in report.blind_spots])

Expected output (matches `data/corpus_facts.md`):

```
coverage_ratio = 0.56
[ ] Northwind Industries — Annual Report FY2025 (0 triples, 4 body lines)  <-- BLIND SPOT
[x] 1. Segment Performance (15 triples, 9 body lines)
[x] 2. Divisions (4 triples, 5 body lines)
[x] 3. Income Statement (8 triples, 7 body lines)
[x] 4. Balance Sheet (3 triples, 6 body lines)
[x] 5. Corporate Facts (4 triples, 6 body lines)
[ ] 6. Management Discussion and Analysis (0 triples, 9 body lines)  <-- BLIND SPOT
[ ] 7. Risk Factors (0 triples, 8 body lines)  <-- BLIND SPOT
[ ] 8. Outlook (0 triples, 5 body lines)  <-- BLIND SPOT

blind spots: ['Northwind Industries — Annual Report FY2025', '6. Management Discussion and Analysis', '7. Risk Factors', '8. Outlook']
```

Roughly half the document's content-bearing regions carry no triple. That is
not a bug — those sections are qualitative prose (MD&A, Risk Factors,
Outlook) that the report itself defers to the tables. The point is that the
gap is **named and counted**, not hidden.

## Listing 1 — a prose question abstains (real Qwen path)

Now the online path. We load the trained store and run a Risk-Factor / MD&A
question through `RagPipeline`. The Outlook and Risk Factors sections have no
triples, so nothing binds — the bind-check fires and the pipeline abstains
with a notice rather than fabricating a paragraph.

> **CI-only cell.** This loads the store and runs Qwen2.5-3B-Instruct on the
> GPU. Do not run it during authoring; the lead executes it in CI.

In [ ]:
# === CI-only (loads store + Qwen on GPU) ===
from knowlytix.knowledge.llm_backend import LocalTransformersBackend
from knowlytix.knowledge.rag import RagConfig, RagPipeline
from knowlytix.knowledge.geode import QWEN_3B

store = load_store()                # exact build geometry GeometryConfig(64,64,32,32), cap loss

qwen = LocalTransformersBackend(QWEN_3B)
rag = RagConfig(llm=qwen)            # bank-grade defaults: dense off, abstain-not-guess
pipe = RagPipeline.from_store(store, rag)

prose_q = "What are the company's main risk factors?"
ans = pipe.query(prose_q)
print("decision:", ans.decision)
print("route:   ", ans.route)
print("notice:  ", ans.notice)
print("answer:  ", ans.answer)
assert ans.decision == "abstain"
assert ans.notice is not None

Expected: `decision: abstain`, `route: triple`, and a notice such as
*"Question did not bind to known entities/relations."* The answer is the
standard refusal string, not a guess. Contrast a chunk-and-pray baseline,
which would happily summarize the Risk Factors prose and present it as an
answer with no way to verify it.

## Listing 3 — the per-query audit trail

Every answer — accepted or abstained — carries a structured audit record:
the decision, the route, the query triples, what bound, the source facts and
their provenance spans, the verification verdicts, and the notice. This is
the bank-grade logging surface; wire it to `RagConfig.audit_sink` to capture
every query automatically.

> **CI-only cell** (uses the `ans` produced above).

In [ ]:
# === CI-only (uses `ans` from Listing 1) ===
import json
audit = ans.audit()
print(json.dumps(audit, indent=2, default=str))
assert audit["decision"] == "abstain"
assert audit["route"] == "triple"
assert audit["verified"] is True   # abstaining IS a verified outcome

The audit shows `decision: abstain` with an empty `sources` list — there was
no grounded fact to cite, which is exactly why the system declined. Note
`verified: true`: an honest abstention is a *verified* outcome, not an
error. Compare this trail to a dense-retrieval answer in Chapter 14, which
carries `verified=False` and a quarantine notice.

## Exercise — clear a blind spot

Add a triple to a blind-spot section and watch coverage clear. There is a
subtlety the library enforces honestly: `coverage_report` only counts a
triple toward a section if its *provenance resolves* into that section. The
`ProvenanceLedger` resolves table cells, section headers (`in_section`), and
schema bullets — a triple invented out of thin air resolves to line `-1`
(`unaligned`) and counts toward nothing. So clearing a blind spot is not a
matter of asserting a fact; the fact must be *anchored* to real source text.

The Outlook section header is itself a resolvable anchor, so we add an
`(outlook, in_section, outlook)` triple. Because `coverage_report` excludes
structural relations by default, we pass `exclude_relations=()` to count it.

In [ ]:
from knowlytix.knowledge.geode.provenance import ProvenanceLedger
ledger = ProvenanceLedger.from_text(markdown)

# An invented content triple does NOT resolve -> it cannot clear a blind spot.
fake = ("cloud platform", "has_outlook", "continued investment")
print("invented triple resolves to line:", ledger.resolve(*fake).line_no,
      "(unaligned)")

# An anchored triple keyed to the real Outlook header DOES resolve.
outlook_triple = ("outlook", "in_section", "outlook")
print("anchored triple resolves to line:", ledger.resolve(*outlook_triple).line_no)

augmented = _StoreView(markdown=markdown,
                       triples=TRIPLES + [outlook_triple],
                       store_path=STORE_PATH)
# Count structural relations too, so the anchored triple registers.
after = coverage_report(augmented, exclude_relations=())
titles = [r.title for r in after.blind_spots]
print("blind spots after:", titles)
print(f"coverage_ratio: {report.coverage_ratio:.2f} -> {after.coverage_ratio:.2f}")
assert "8. Outlook" not in titles

`8. Outlook` drops out of `blind_spots` and the coverage ratio rises
(0.56 → 0.67). The lesson is twofold. First, the remedy for a blind spot is
*more anchored triples* — re-ingest the section, broaden extraction — not
turning on distrusted dense retrieval (the opt-in escape hatch in
Chapter 14, flagged `verified=False` for a reason). Second, the coverage
monitor cannot be gamed: an unanchored claim resolves to `-1` and clears
nothing, so the ratio only moves when real evidence backs the new triple.

## Honest limits

The coverage monitor measures *triple presence per section*, not *answer
quality*. A section with triples can still fail to answer a specific
question (the triple exists but the asked attribute differs — that is the
relevance gate's job, Chapter 10), and a covered section's triples can be
wrong if extraction erred (that is GEODE's job, Chapter 5). Coverage also
says nothing about whether a blind spot *should* have been triplified —
qualitative prose like Risk Factors legitimately carries no authoritative
fact, so a 0.56 ratio is not a defect to drive to 1.0. Finally, abstention
here is binary: the pipeline does not rank *how close* it came to binding,
so a near-miss paraphrase and a wholly off-topic question both simply
abstain. Calibrating the bind threshold so paraphrases resolve while
off-topic queries still abstain is Chapter 12.

## Self-check

The chapter's claim: the prose sections are measurable blind spots, and the
coverage monitor names them. This runs CPU-only against the real library
(no store, no Qwen).

In [ ]:
blind = {r.title for r in report.blind_spots}
assert "7. Risk Factors" in blind
assert "8. Outlook" in blind
assert "6. Management Discussion and Analysis" in blind
# The fact-bearing tables are NOT blind spots.
assert "1. Segment Performance" not in blind
assert "3. Income Statement" not in blind
assert abs(report.coverage_ratio - 0.56) < 0.01
print("OK: prose sections are measurable blind spots; tables are covered.")